# California housing with pipelines

The problem of [California housing, step by step](california_by_hand.ipynb), on the version of the data in Aurélien Géron's book: the same 20,640 districts, with counts per district instead of averages per household, and two problems added on purpose for teaching, a text column and missing values. Each preparation step is a scikit-learn transformer, and all of them, with the model at the end, become a single object that is fitted with one call and predicts from a raw district. Models are compared on a validation set and tested once, as in [Validation and model selection](../ml-fundamentals/validation_and_model_selection.ipynb).

## The raw data

This is the version of the dataset that accompanies Aurélien Géron's *Hands-On Machine Learning*, read straight from the book's repository:

In [ ]:
import pandas as pd

housing = pd.read_csv("https://github.com/ageron/data/raw/main/housing/housing.csv")
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [ ]:
housing.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.7 MB


In [ ]:
housing["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

Three differences from the version of [California housing, step by step](california_by_hand.ipynb):

- `total_bedrooms` is missing in 207 districts: 20,433 values for 20,640 rows.
- `ocean_proximity` is text, with five categories. One of them, `ISLAND`, has only five districts, so a split can leave it out of the training rows altogether: a category that only new data will bring.
- Rooms, bedrooms, people and households are **totals for the whole district**, not averages per household.

The target is in dollars here, capped in the same way, at \$500,001.

Each of these problems is explained in general, beyond this dataset, in [Missing values](../../data-preparation/missing_values.ipynb), [Encoding categories](../../data-preparation/categorical_encoding.ipynb) and [Creating features](../../data-preparation/deep-dives/creating_features.ipynb); here they are only solved.

## Splitting

A test set is held back first, and the rest is split into training and validation sets, as in [Validation and model selection](../ml-fundamentals/validation_and_model_selection.ipynb):

In [ ]:
from sklearn.model_selection import train_test_split

X = housing.drop(columns="median_house_value")
y = housing["median_house_value"]
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_validation, y_train, y_validation = train_test_split(X_rest, y_rest, test_size=0.25, random_state=42)
len(X_train), len(X_validation), len(X_test)

(12384, 4128, 4128)

## A pipeline of transformers

By hand, every preparation step has to be fitted on the training rows and then applied, in the same order and with the same learned values, to the validation rows, the test rows and every future district. With several steps this is easy to get wrong, and the mistake is silent: a `fit_transform` called on the validation set by accident doesn't fail, it just leaks.

A `Pipeline` holds the steps in order. Its `fit` calls `fit_transform` on every step but the last, passing each one's output to the next, and `fit` on the last, so that every step learns from the training rows only; its `transform` applies all of them with what they learned. For the numeric columns, two steps: fill the missing values with the median of each column, then standardise:

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numeric_columns = X.columns.drop("ocean_proximity")
numeric_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
numeric_pipeline.fit(X_train[numeric_columns])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('simpleimputer', ...), ('standardscaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['longitude','latitude','housing_median_age',...,'population','households', 'median_income']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for 

`make_pipeline` names each step after its class; `Pipeline([("impute", ...), ("scale", ...)])` names them explicitly. Each step can be reached by its name, with what it learned from the training rows:

In [ ]:
pd.Series(numeric_pipeline.named_steps["simpleimputer"].statistics_, index=numeric_columns)

longitude             -118.48000
latitude                34.25000
housing_median_age      29.00000
total_rooms           2126.00000
total_bedrooms         437.00000
population            1172.00000
households             410.00000
median_income            3.51695
dtype: float64

`transform` returns a NumPy array, without the column names; `get_feature_names_out` gives them back (or `set_output(transform="pandas")` makes every step return a DataFrame). Every value is now standardised, and none is missing:

In [ ]:
prepared = pd.DataFrame(
    numeric_pipeline.transform(X_validation[numeric_columns]),
    columns=numeric_pipeline.get_feature_names_out(),
    index=X_validation.index,
)
prepared.isna().sum().sum(), prepared.shape

(np.int64(0), (4128, 8))

## Different steps for different columns

`ocean_proximity` is text, and needs a different treatment: one-hot encoding turns it into one 0/1 column per category. `handle_unknown="ignore"` decides what happens to a category that appears only in new data: all its columns are 0, instead of an error at prediction time.

A `ColumnTransformer` applies one transformer, or one pipeline, to each group of columns, and joins their outputs side by side:

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

basic_preprocessing = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("category", OneHotEncoder(handle_unknown="ignore"), ["ocean_proximity"]),
])
basic_preprocessing.fit(X_train)
basic_preprocessing.get_feature_names_out()

array(['numeric__longitude', 'numeric__latitude',
       'numeric__housing_median_age', 'numeric__total_rooms',
       'numeric__total_bedrooms', 'numeric__population',
       'numeric__households', 'numeric__median_income',
       'category__ocean_proximity_<1H OCEAN',
       'category__ocean_proximity_INLAND',
       'category__ocean_proximity_ISLAND',
       'category__ocean_proximity_NEAR BAY',
       'category__ocean_proximity_NEAR OCEAN'], dtype=object)

## Steps scikit-learn doesn't have

The raw columns count rooms, bedrooms and people *per district*, so a large district has many rooms simply because it is large. What describes the houses is a ratio: rooms per household, bedrooms per room, people per household. These are the averages that [California housing, step by step](california_by_hand.ipynb) received ready-made.

A ratio learns nothing from the data, so any function can compute it. `FunctionTransformer` turns a function into a transformer that fits in a pipeline. `feature_names_out` tells it how to name its output, so that the names survive the pipeline:

In [ ]:
from sklearn.preprocessing import FunctionTransformer


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(transformer, input_names):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )

`column_ratio` is a named function, not a `lambda`: a fitted pipeline is saved to a file to be used later, and Python can't save a `lambda` that way. Even a named function is saved by its name only: the program that loads the pipeline must be able to import it, so in a project it lives in a module, not in a notebook.

The full preparation: three ratios, the categories, and the remaining numeric columns through the numeric pipeline (`remainder`):

In [ ]:
preprocessing = ColumnTransformer(
    [
        ("bedrooms_per_room", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_household", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_household", ratio_pipeline(), ["population", "households"]),
        ("category", OneHotEncoder(handle_unknown="ignore"), ["ocean_proximity"]),
    ],
    remainder=make_pipeline(SimpleImputer(strategy="median"), StandardScaler()),
)
preprocessing.fit(X_train)
preprocessing.get_feature_names_out()

array(['bedrooms_per_room__ratio', 'rooms_per_household__ratio',
       'people_per_household__ratio',
       'category__ocean_proximity_<1H OCEAN',
       'category__ocean_proximity_INLAND',
       'category__ocean_proximity_ISLAND',
       'category__ocean_proximity_NEAR BAY',
       'category__ocean_proximity_NEAR OCEAN', 'remainder__longitude',
       'remainder__latitude', 'remainder__housing_median_age',
       'remainder__median_income'], dtype=object)

Each output name starts with the name of the step that produced it. `remainder` only covers the columns that no step names: longitude, latitude, age and income. The four counts reach the models only through their ratios; keeping them as well would be one more line in the list.

## A model as the last step

When the last step of a pipeline is a predictor, the whole pipeline is a model: `fit` prepares the training rows and trains, and `predict` prepares new rows *with what was learned from the training rows* and predicts. The four kinds of model of [Validation and model selection](../ml-fundamentals/validation_and_model_selection.ipynb), each behind its own copy of the preparation (`clone` copies it without anything learned, so that no two models share a fitted step):

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

models = {
    "linear regression": make_pipeline(clone(preprocessing), LinearRegression()),
    "k-nearest neighbours": make_pipeline(clone(preprocessing), KNeighborsRegressor(n_neighbors=10)),
    "decision tree": make_pipeline(clone(preprocessing), DecisionTreeRegressor(max_depth=10, random_state=42)),
    "random forest": make_pipeline(clone(preprocessing), RandomForestRegressor(random_state=42, n_jobs=-1)),
}
for model in models.values():
    model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error

validation_errors = pd.DataFrame({
    name: {
        "training MAE ($)": mean_absolute_error(y_train, model.predict(X_train)),
        "validation MAE ($)": mean_absolute_error(y_validation, model.predict(X_validation)),
    }
    for name, model in models.items()
}).T.round(0)
validation_errors

,training MAE ($),validation MAE ($)
linear regression,51866.0,50757.0
k-nearest neighbours,37758.0,40692.0
decision tree,30376.0,39212.0
random forest,12095.0,31624.0


The same ranking as in [Validation and model selection](../ml-fundamentals/validation_and_model_selection.ipynb), with every validation error a little lower: about \$31,600 for the forest against \$33,500 there, and \$50,800 for the linear model against \$53,300. Part of the reason is the category, which states directly what latitude and longitude only imply: whether a district is inland or by the sea. The two notebooks don't hold out the same districts, though, so differences of a couple of thousand dollars are within what a different validation set alone could produce.

In [ ]:
chosen_name = validation_errors["validation MAE ($)"].idxmin()
final_model = clone(models[chosen_name]).fit(X_rest, y_rest)
mean_absolute_error(y_test, final_model.predict(X_test))

32386.308803294574

The forest is chosen again, retrained on the training and validation rows, and tested once: it misses the value of a test district by about \$32,400 on average, against about \$53,300 for the linear model of [California housing, step by step](california_by_hand.ipynb).

## One object for production

The fitted pipeline receives districts exactly as they arrive in the raw file, text and missing values included, and returns a price. Nothing has to be prepared by hand before a prediction, so nothing can be prepared differently from the training rows. Three districts from the test set, raw:

In [ ]:
new_districts = X_test.head(3)
new_districts

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
20046,-122.38,40.67,10.0,2281.0,444.0,1274.0,438.0,2.2120,INLAND
3024,-118.37,33.83,35.0,1207.0,207.0,601.0,213.0,4.7308,<1H OCEAN
15663,-117.24,32.72,39.0,3089.0,431.0,1175.0,432.0,7.5925,NEAR OCEAN


In [ ]:
final_model.predict(new_districts).round(-3)

array([ 90000., 312000., 409000.])

That single object is what gets saved to a file and served to other programs: *Serving models* *(coming)*, in the PIA module, takes it from here. Its inside can be inspected like any pipeline, step by step:

In [ ]:
final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('randomforestregressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['longitude','latitude','housing_median_age',...,'households', 'median_income','ocean_proximity']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('bedrooms_per_room', ...), ('rooms_per_household', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the 

## Three meanings of "pipeline" *(deep dive)*

The word is used at three levels, and the context says which one:

- **The `Pipeline` object** of this notebook: a chain of transformers ending, usually, in a model; one object that is fitted and predicts.
- **The training pipeline:** the whole sequence that produces a model, from loading the data to the final test, as in this notebook. Part of it can't live inside a `Pipeline` object: the split, the comparison of models, the decision. What runs for each new prediction is only the fitted `Pipeline` object, sometimes called the *inference pipeline*: much lighter than the training process that produced it.
- **The MLOps pipeline:** the same sequence automated and run again on a schedule or when the data changes, with deployment and monitoring. It belongs to the operation of a system in production, beyond this topic.

What happens inside a `Pipeline`, and why every scikit-learn object fits in one, is the subject of the deep dive *scikit-learn from scratch* *(coming)*.